# 리포트 07 — 마이크로도플러: 도는 로터가 만드는 무늬를 무엇이 정하는가

> ### 한 일
> **로터를 돌려가며 매 시간표본마다 다시 추적해 슬로타임 복소열을 만들고, 회전수·가림·자세 셋을 각각 단일축으로 흔들어 무늬가 어떻게 바뀌는지를 쟀다.**

### 결과
1. 네 로터를 **같은 회전수로** 돌리면 반창 스펙트럼 상관이 0.9966 ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.findings.rpm_spread_makes_it_time_varying.locked_half_corr⟩ 다 — 신호가 완전한 주기함수라 스펙트로그램이 창 내내 같은 모습으로 선다. 로터마다 흩뜨리면 0.7757 ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.findings.rpm_spread_makes_it_time_varying.spread_half_corr⟩ 로 내려가고 그때 줄무늬가 숨쉰다.
2. 동체가 날개를 가리면 변조 깊이가 -8.69 dB ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.findings.occlusion_ptp_db⟩, 레벨이 +1.13 dB ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.findings.occlusion_level_db⟩ 바뀐다 — 광선 엔진·재질·기하·운동학·광선 격자를 전부 같게 두고 «동체가 막느냐» 만 다르게 한 값이다.
3. 프로펠러가 동체 위에 있어 **위에서 보면 날개가 통째로 드러난다**. 지상 레이더는 기체를 아래에서 보므로 앙각이 음수다 — 그래서 이 편의 자세는 배 쪽이다.
4. 자세×로터위상 1,152 칸 ⟨outputs/report00_microdoppler.json : specular_census.total.n_cells⟩ 전수에서 프로펠러에 떨어진 정반사 경로는 0 칸 ⟨outputs/report00_microdoppler.json : specular_census.total.n_with_prop_specular⟩ 이다 — 위상은 광선 엔진이, 세기는 PO 커널이 맡는 이유가 여기 있다.
5. ⚠ 상시 기준신호로는 **날개끝 도플러를 못 본다**. 필요 표본율이 1980 Hz ⟨outputs/md_range_sweep.json : cells[0].prf_feasibility.LTE CRS.required_prf_hz⟩ 인데 LTE CRS 는 1000 Hz ⟨outputs/md_range_sweep.json : cells[0].prf_feasibility.LTE CRS.mode_prf_hz⟩ 다. 살아남는 것은 **블레이드 통과율(플래시선)** 뿐이다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 슬로타임 복소열 | 시간표본마다 로터 위상을 다시 놓고 광선을 다시 쏜다 — 위상 하나짜리 표를 쓰지 않으므로 로터마다 회전수를 다르게 줄 수 있다 |
| 무엇이 그것을 가능하게 했나 | `src/articulated_fast.py` — 드론을 한 번 짓고 위상마다 행렬곱만 한다. 정점 배열이 옛 함수와 비트 단위로 같다 |
| 가림 단일축 | 한쪽은 동체를 완전흡수(Γ=0)로 두어 막되 산란은 안 하게 하고, 다른 쪽은 동체 면만 빼되 정점은 남겨 광선 격자를 같게 유지한다 |
| 도플러 분해능 | 창에 든 블레이드 주기 수가 정한다. 표본 수를 늘려도 안 좋아진다 |
| 헤드라인 기체 선택 | DJI Matrice 4E — 프롭·벨 겹침이 0.01 % 로 정리됐고 1차 실측 표적이다. Mini 5 Pro 는 겹침이 남아 있어 따로 싣는다 |

### 재현

```bash
PYTHONPATH=src python benchmark/report15b_microdoppler_recompute.py
PYTHONPATH=src python benchmark/report15b_stamp_provenance.py
PYTHONPATH=src python benchmark/build_report15b_figs.py
PYTHONPATH=src python src/make_report07_microdoppler.py
```

| | |
|---|---|
| 출력 | `outputs/report15b_microdoppler.json`, `outputs/report15b_series.npz` |
| 소요 | 약 25분 (GPU 1장 — 광선 추적이 6칸 × 4팔) |
| 비고 | 산출물이 자기가 어떤 메쉬로 계산됐는지 지문을 함께 적는다(`mesh_provenance`) — 계산 도중 메쉬가 바뀌면 스스로 경고한다 |

---

## §1. 무엇을 재는가 — 슬로타임 복소열

표적이 제자리에 떠 있어도 날개는 돈다. 날개 표면의 점들이 시간에 따라 자리를 바꾸므로 왕복 위상이 변조되고, 그것이 되돌아오는 신호의 느린 시간축에 실린다. 우리는 그 열을 **시간표본마다 자세를 새로 놓고 광선을 다시 쏘아** 만든다.

왜 그렇게까지 하는가. 로터마다 회전수를 다르게 주려면 드론 전체 자세가 각도 하나의 함수가 아니게 되고, 그러면 «위상 하나짜리 표를 미리 만들어 두고 조회한다» 는 지름길이 막힌다. 조립을 싸게 만들어 그 지름길을 버렸다.

이 편의 헤드라인 칸은 DJI Matrice 4E ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.name⟩ 를 배 쪽에서 본 것이다 — 방위 0 도 ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.az_deg⟩ · 앙각 -15 도 ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.el_deg⟩, 호버 3800 rpm ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.physics.rpm⟩, 운동학이 예측하는 날개끝 주파수 1230 Hz ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.physics.f_tip⟩, 블레이드 통과율 126.7 Hz ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.physics.f_flash⟩.

## §1a. 두 엔진이 같은 로터를 돌린다

![report07_f2](outputs/figures/report07_f2.png)

**그림 2.** Sionna 자체 엔진과 우리 PO 커널이 같은 로터에서 같은 무늬를 내는가?
로터 위상을 스텝하고 매번 다시 추적하는 같은 절차를 **서로 다른 두 엔진**에 태웠다 — 하나는 Sionna 의 PathSolver 이고 하나는 우리 PO 커널이다. 운동학이 예측한 날개끝 주파수 **아래**에서 두 빗살이 겹친다.

⭐ 그 위에서 갈린다. Sionna 의 꼬리가 우리 것보다 20~30 dB 높게 남는다. 그 꼬리는 블레이드가 만든 것이 아니므로, 가장자리를 자동으로 찾는 검출기는 물리적이지 않은 자리를 가장자리라고 보고한다.

⭐ 두 엔진이 아래쪽에서 겹친다는 것이 «위상은 광선 엔진이 맞게 낸다» 의 근거이고, 위쪽에서 갈린다는 것이 «세기는 PO 커널이 맡는다» 의 근거다.

## §1b. 전처리를 어떻게 했는가

마이크로도플러 그림은 전처리가 답을 바꾼다. 그래서 규약을 적어 둔다.

| 단계 | 우리가 한 것 | 왜 |
|---|---|---|
| 채널 | 전체 드론과 프로펠러만을 따로 | 동체가 블레이드를 덮는다(§5) |
| 0 도플러 | **살린다** | 동체 선이 읽기의 기준이다 — 선행 그림 흐름을 따랐다 |
| 조각 길이 | 블레이드 13 주기 | 능선 사이에 13 빈이 들어 빗살이 안 뭉갠다 |
| 창·제로패딩 | Hann · 4배 | 누설을 줄이고 주파수축을 매끈하게 |
| 색역 | 60 dB | 동체 선을 0 dB 로 두고 능선을 그 아래에서 읽는다 |
| 정규화 | 한 그림 안에서 공통 | 두 패널을 나란히 놓고 비교할 수 있게 |
⭐ 정적 성분을 지우는 슬로타임 고역통과(MTI)는 `src/microdoppler_proc.py` 에 따로 있다 — **검출 축**에서 쓴다. ⚠ 그 노치는 호버하는 표적의 동체도 함께 지우므로 탐지에서는 대가가 된다.

⚠ 선행 구현의 **처리 파라미터**는 그 시스템의 자원격자에 맞춰진 값이라 그대로 옮기지 않았다. 우리가 가져온 것은 그림을 읽는 순서이고, 차단주파수 같은 것은 우리 물리에서 정했다.

## §2. 회전수가 같으면 무늬는 시간에 못 변한다

![report07_f1](outputs/figures/report07_f1.png)

**그림 1.** 네 로터가 같은 회전수로 돌 때와 흩어질 때, 무늬가 시간에 따라 어떻게 다른가?
왼쪽은 네 로터를 같은 회전수로 돌린 것이다. 줄무늬가 시간축 내내 **같은 자리에 선다**. 창을 반으로 갈라 두 스펙트럼의 상관을 재면 0.9966 ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.findings.rpm_spread_makes_it_time_varying.locked_half_corr⟩ 다. 우연이 아니라 **원리**다 — 네 로터가 같은 속도로 위상까지 맞춰 돌면 신호가 완전한 주기함수가 되고, 주기함수의 스펙트로그램은 창 내내 자기 모습을 지킨다.

오른쪽은 로터마다 회전수를 2% ⟨outputs/report15b_microdoppler.json : _meta.rpm_spread_frac⟩ 흩뜨린 것이다. 상관이 0.7757 ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.findings.rpm_spread_makes_it_time_varying.spread_half_corr⟩ 로 내려가고 줄이 숨쉬듯 흔들린다.

⚠ 흩어짐 폭은 **선언된 가정**이다 — 실측 비행 로그는 앞으로 확보한다. 실제 기체는 무게중심 치우침과 요 토크 균형 때문에 네 모터가 서로 다른 추력을 내고 그만큼 회전수가 갈린다. 그 폭은 우리 표적에서 측정 대기 상태다. 근거는 `outputs/report15b_microdoppler.json : _meta.spread_is_declared_ko` 에 적었다.

## §3. 동체가 날개를 가리면

![report07_f4](outputs/figures/report07_f4.png)

**그림 3.** 같은 광선·같은 메쉬·같은 운동에서 동체가 막으면 무엇이 달라지는가?
가림만 남기고 다른 것을 전부 묶었다. 한쪽은 동체를 완전흡수로 두어 **광선은 막되 산란은 안 하게** 하고, 다른 쪽은 동체 면만 빼되 정점 배열은 남겨 두었다 — 정점을 남기면 경계상자가 같아서 두 팔의 광선 수와 간격이 같아진다. 정점을 빼면 «가림» 과 «표본화» 가 섞인다.

결과는 변조 깊이 -8.69 dB ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.findings.occlusion_ptp_db⟩ · 레벨 +1.13 dB ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.findings.occlusion_level_db⟩ 다. 같은 그림의 오른쪽에서 운동학이 예측한 날개끝 주파수 안쪽에 빗살이 서고 바깥에서 떨어지는 것도 읽힌다.

⚠ 부호를 물리로 단정하지 않는다. 합이 코히런트라 항이 줄어도 남은 항끼리 상쇄가 덜 되면 레벨이 **올라갈 수 있다** — 실제로 그런 칸이 있다. 근거는 `outputs/report15b_microdoppler.json : cells.*.findings.occlusion_sign_note_ko`.

## §4. 자세가 답을 바꾼다

![report07_f3](outputs/figures/report07_f3.png)

**그림 4.** 블레이드 신호는 약한가, 아니면 동체가 덮고 있는가?
프로펠러는 동체 **위**에 있다. 그래서 위에서 내려다보면 날개가 통째로 드러나고, 그 자세에서 가림은 0 에 가깝다. 지상 레이더는 비행 중인 기체를 **아래에서** 보므로 기체 좌표계로 앙각이 음수이고, 거기서 가림이 문다.

이것은 우리 자유공간 명세가 이미 적어 둔 것과 같은 결론이다 — 자유공간 바이스태틱의 이등분선 앙각은 전 구간 음수라 우리는 드론 배를 본다. 자세 스윕이 그 문장에 독립적인 근거를 붙였다.

⚠ Mini 5 Pro 는 프로펠러와 모터 벨이 겹친 삼각형이 남아 있어 헤드라인에서 뺐다. 같은 그림을 그 기체로도 그려 `outputs/figures/report15b_f1b.png` · `report15b_f2b.png` 에 뒀다 — 겹침이 정리되기 전 값이라는 것을 알고 읽어야 한다.

## §4a. 블레이드는 강하다 — 동체가 덮고 있을 뿐이다

![report07_f3](outputs/figures/report07_f3.png)

**그림 4.** 블레이드 신호는 약한가, 아니면 동체가 덮고 있는가?
같은 자세·같은 주파수에서 전체 드론과 프로펠러 채널을 따로 쟀다. 3.5 GHz 에서 전체 드론의 변조는 **2.84 dB**, 프로펠러만은 **32.32 dB** 다. ⭐ 블레이드 신호가 약한 것이 아니라 **동체 정적 반사가 덮고 있다**.

이것이 전처리에서 정적 성분을 지우는 이유다. 지우고 나면 블레이드가 30 dB 대로 드러난다.

⚠ 주파수를 올려도 프로펠러 채널의 변조는 21~36 dB 대에 머문다. 우리 대역에서 블레이드 폭은 파장의 0.09~0.24 배로 전기적으로 작은데, 그래도 변조 자체는 충분히 크다. 어려운 것은 «블레이드가 약하다» 가 아니라 «동체와 블레이드를 가르는 일» 이다.

## §5. 상시 신호로는 어디까지 보이는가

패시브 레이더는 남이 쏘는 신호를 쓴다. 그 신호가 얼마나 자주 반복되는지가 우리가 볼 수 있는 도플러의 상한을 정한다. 날개끝 도플러를 접힘 없이 보려면 표본율이 그 두 배는 돼야 하는데, 1980 Hz ⟨outputs/md_range_sweep.json : cells[0].prf_feasibility.LTE CRS.required_prf_hz⟩ 가 필요한 자리에 LTE CRS 는 1000 Hz ⟨outputs/md_range_sweep.json : cells[0].prf_feasibility.LTE CRS.mode_prf_hz⟩, 5G SSB 는 50 Hz ⟨outputs/md_range_sweep.json : cells[0].prf_feasibility.5G SSB.mode_prf_hz⟩ 다.

⚠ 그래서 상시 신호가 주는 것은 **블레이드 통과율까지**다. 헤드라인 기체의 126.7 Hz ⟨outputs/report15b_microdoppler.json : cells.matrice4e/belly.physics.f_flash⟩ 는 LTE·WiFi 의 나이퀴스트 안에 든다. 5G 쪽 두 모드에서는 그 통과율마저 접힌다.

⭐ 이 구분이 실험 설계를 바꾼다. 날개끝 확산을 보려면 기준 안테나가 **풀 파형을 받아야** 하고, 그건 «상시 신호만 쓴다» 와 다른 조건이다. 통과율만으로 무엇을 할 수 있는지가 별도 질문으로 남는다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 비행 로그의 모터별 회전수를 넣는다 | 지금 선언된 가정으로 둔 흩어짐 폭이 측정값으로 바뀐다 | 실측 1차 · Matrice 4E |
| 가림을 자세 전면으로 넓힌다 | 어느 자세에서 얼마나 무는지의 지도가 서고, 그 지도가 분류기의 입력이 된다 | 이 편 §3 을 격자로 |
| 통과율만 보이는 조건에서 탐지·분류를 돌린다 | 상시 신호만으로 어디까지 가는지가 수치로 갈린다 | 이 편 §5 |
| 지면 반사가 만드는 선을 블레이드 선과 대조한다 | 환경이 이 무늬를 훼손하는지, 가짜 선이 구별되는지가 정해진다 | 야외 사이트 트윈 |